In [ ]:
!pip install -U transformers trl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=deb73ac4338db56ed22060c9dd8049198d20d05a12460eaf17da5c3b4bf34eeb
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
# =========================================================
# 1. Install
# =========================================================
!pip install unsloth transformers datasets evaluate trl -q

from transformers import EarlyStoppingCallback

# =========================================================
# 2. Imports
# =========================================================
import torch
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer

# =========================================================
# 3. Dataset Processor (OOP)
# =========================================================
class DatasetProcessor:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path).head(12000)   # ✅ only 500 rows

    def format_data(self):
        data = []
        for _, row in self.df.iterrows():
            if pd.isna(row["Questions"]) or pd.isna(row["Answers"]):
                continue

            data.append({
                "text": f"### Instruction:\n{row['Questions']}\n\n### Response:\n{row['Answers']}"
            })
        return Dataset.from_list(data)

# =========================================================
# 4. Strategy Pattern (LoRA)
# =========================================================
class LoRAStrategy:
    def apply(self, model):
        return FastLanguageModel.get_peft_model(
            model,
            r=32,
            target_modules=["q_proj","k_proj","v_proj","o_proj"],
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            use_gradient_checkpointing=True,
        )

# =========================================================
# 5. Load Model (Unsloth, 4-bit)
# =========================================================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3.1-8b-instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = LoRAStrategy().apply(model)

# =========================================================
# 6. Load Dataset
# =========================================================

processor = DatasetProcessor(
    "/content/drive/MyDrive/BengaliEmpatheticConversationsCorpus  - Copy.csv"
)

dataset = processor.format_data()

dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# =========================================================
# 7. FineTuner
# =========================================================
class LLAMAFineTuner:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def train(self, train_dataset, eval_dataset):
        training_args = TrainingArguments(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=16,
            num_train_epochs=3,
            learning_rate=2e-5,

            fp16=True,          # ✅ Kaggle compatible
            bf16=False,

            logging_steps=10,

            eval_strategy="steps",   # ✅ FIXED
            eval_steps=50,

            save_strategy="steps",
            save_steps=50,

            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,

            remove_unused_columns=False,   # ✅ important for SFTTrainer

            optim="adamw_8bit",            # ✅ faster + less VRAM

            output_dir="outputs",
            report_to="none"
        )

        trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            dataset_text_field="text",
            max_seq_length=2048,
            args=training_args,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        trainer.train()
        return trainer

# =========================================================
# 8. Train
# =========================================================
trainer = LLAMAFineTuner(model, tokenizer).train(train_dataset, eval_dataset)

==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Current model requires 256 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/10779 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/1198 [00:00<?, ? examples/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 219.81 MiB is free. Including non-PyTorch memory, this process has 14.35 GiB memory in use. Of the allocated memory 14.07 GiB is allocated by PyTorch, and 127.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
trainer.model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

In [ ]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# =========================================================
# 9. Evaluator (FIXED)
# =========================================================
import evaluate
import math

class Evaluator:
    def __init__(self):
        self.bleu = evaluate.load("bleu")
        self.rouge = evaluate.load("rouge")

    def compute_metrics(self, predictions, references):
        bleu = self.bleu.compute(
            predictions=predictions,
            references=references
        )

        rouge = self.rouge.compute(
            predictions=predictions,
            references=[ref[0] for ref in references]
        )

        return {"bleu": bleu, "rouge": rouge}

    def compute_perplexity_from_logs(self, trainer):
        eval_loss = None

        for log in reversed(trainer.state.log_history):
            if "eval_loss" in log:
                eval_loss = log["eval_loss"]
                break

        return math.exp(eval_loss) if eval_loss is not None else None

# =========================================================
# 10. Helper Functions (NEW)
# =========================================================
def extract_response(text):
    if "### Response:" in text:
        return text.split("### Response:")[-1].strip()
    return text.strip()


def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# =========================================================
# 11. Evaluation Pipeline (FIXED)
# =========================================================
# Reload the trained model and tokenizer for evaluation
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="lora_model", # Path to your saved LoRA weights
    max_seq_length=2048,     # Ensure this matches training or is sufficient
    dtype=None,              # Autodetect dtype
    load_in_4bit=True,
)

model.eval()
FastLanguageModel.for_inference(model)

predictions = []
references = []
responses_log = []

for i, sample in enumerate(eval_dataset):
    full_text = sample["text"]

    # Split prompt & reference
    if "### Response:" not in full_text:
        continue

    prompt = full_text.split("### Response:")[0]
    ref = full_text.split("### Response:")[1]

    pred = generate_response(model, tokenizer, prompt)

    # Clean outputs
    pred_clean = extract_response(pred)
    ref_clean = extract_response(ref)

    # Avoid empty strings (BLEU crash fix)
    if pred_clean == "":
        pred_clean = " "
    if ref_clean == "":
        ref_clean = " "

    predictions.append(pred_clean)
    references.append([ref_clean])  # BLEU format

    # Save logs
    responses_log.append({
        "prompt": prompt,
        "prediction": pred_clean,
        "reference": ref_clean
    })


# Save Generated Responses
pd.DataFrame(responses_log).to_csv("GeneratedResponses.csv", index=False)


# =========================================================
# 12. Metrics (FINAL)
# =========================================================
evaluator = Evaluator()

metrics = evaluator.compute_metrics(predictions, references)
perplexity = evaluator.compute_perplexity_from_logs(trainer)

metrics["perplexity"] = perplexity
print("✅ FINAL METRICS:")
print(metrics)


# =========================================================
# 13. LLAMAExperiments Log (SAFE)
# =========================================================
from datetime import datetime

train_loss = None
val_loss = None

for log in trainer.state.log_history:
    if "loss" in log:
        train_loss = log["loss"]
    if "eval_loss" in log:
        val_loss = log["eval_loss"]

train_loss = float(train_loss) if train_loss is not None else None
val_loss = float(val_loss) if val_loss is not None else None

experiment_log = {
    "id": 1,
    "model_name": "llama-3.1-8b-instruct",
    "lora_config": {
        "r": 16,
        "alpha": 16,
        "dropout": 0
    },
    "train_loss": train_loss,
    "val_loss": val_loss,
    "metrics": metrics,
    "timestamp": str(datetime.now())
}

pd.DataFrame([experiment_log]).to_csv("LLAMAExperiments.csv", index=False)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
torch._dynamo.disable()

In [ ]:
# =========================================================
# 1. INSTALL
# =========================================================
!pip install -q unsloth transformers datasets trl evaluate accelerate bitsandbytes

# =========================================================
# 2. IMPORTS
# =========================================================
import torch, gc
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
import evaluate

# =========================================================
# 3. LOAD DATA
# =========================================================
df = pd.read_csv("/content/drive/MyDrive/BengaliEmpatheticConversationsCorpus  - Copy.csv")

df = df.rename(columns={"Questions": "input", "Answers": "response"})
df = df[['input', 'response']].dropna()
df = df.drop_duplicates()
df = df[df['response'].str.len() > 5]

print("Dataset size:", len(df))

# =========================================================
# 4. SAFE SAMPLER
# =========================================================
def smart_sample(data, n):
    return data.sample(n=min(n, len(data)), random_state=42)

# =========================================================
# 5. FORMAT (FIXED SAFE TOKEN LENGTH)
# =========================================================
MAX_LEN = 256

def format_data(example):
    inp = str(example["input"])[:120]
    res = str(example["response"])[:120]

    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are a helpful Bangla empathetic assistant.\n\n"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{inp}\n\n"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        f"{res}"
    )

    return {"text": text[:MAX_LEN]}

def prepare_dataset(df_):
    ds = Dataset.from_pandas(df_)
    ds = ds.map(format_data, remove_columns=ds.column_names)
    return ds.train_test_split(test_size=0.1, seed=42)

# =========================================================
# 6. MODEL LOAD (STABLE)
# =========================================================
def load_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/llama-3.2-3b-instruct",
        max_seq_length=256,
        load_in_4bit=True,
        dtype=torch.float16,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=8,
        target_modules=["q_proj","k_proj","v_proj","o_proj"],
        lora_alpha=16,
        lora_dropout=0.05,
        use_gradient_checkpointing=True,
    )

    return model, tokenizer

# =========================================================
# 7. TRAIN FUNCTION (FIXED IMPORTANT BUGS)
# =========================================================
def train_stage(dataset, output_dir, model, tokenizer):

    args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,

        num_train_epochs=1.5,
        learning_rate=2e-5,

        fp16=True,
        logging_steps=20,

        eval_strategy="steps",   # ✅ FIXED
        eval_steps=100,

        save_steps=100,
        save_total_limit=2,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        report_to="none"
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        dataset_text_field="text",

        max_seq_length=256,

        packing=False,   # ✅ IMPORTANT FIX (prevents cross_entropy/Dynamo issues)

        args=args
    )

    trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=2))

    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    return model, tokenizer

# =========================================================
# 8. CURRICULUM DATA (SAFE)
# =========================================================
df_5k  = smart_sample(df, 5000)
df_15k = pd.concat([df_5k, smart_sample(df.drop(df_5k.index), 10000)])
df_35k = pd.concat([df_15k, smart_sample(df.drop(df_15k.index), 20000)])

# =========================================================
# 9. TRAIN PIPELINE (NO RELOAD BUG)
# =========================================================
model, tokenizer = load_model()

dataset_5k = prepare_dataset(df_5k)
model, tokenizer = train_stage(dataset_5k, "./stage_5k", model, tokenizer)

gc.collect(); torch.cuda.empty_cache()

dataset_15k = prepare_dataset(df_15k)
model, tokenizer = train_stage(dataset_15k, "./stage_15k", model, tokenizer)

gc.collect(); torch.cuda.empty_cache()

dataset_35k = prepare_dataset(df_35k)
model, tokenizer = train_stage(dataset_35k, "./final_model", model, tokenizer)

# =========================================================
# 10. SIMPLE TEST
# =========================================================
def test_one(model, tokenizer, text):
    prompt = f"""<|start_header_id|>user<|end_header_id|>
{text}

<|start_header_id|>assistant<|end_header_id|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    out = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(out[0], skip_special_tokens=True)

print(test_one(model, tokenizer, "আমি খুব দুঃখিত 😔"))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.6/419.6 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.

model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/31500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/3500 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 31,500 | Num Epochs = 2 | Total steps = 5,907
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 4,587,520 of 3,217,337,344 (0.14% trained)


Step,Training Loss,Validation Loss
100,0.685319,0.669711
200,0.685313,0.666514
300,0.676610,0.664789
400,0.680989,0.664135
500,0.679004,0.662849
600,0.678306,0.660266
700,0.667375,0.659423
800,0.655644,0.657376
900,0.669413,0.656538
1000,0.670718,0.655857


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

In [ ]:
!pip install -U transformers accelerate -q
import transformers
print(transformers.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.4.4 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.5.4 which is incompatible.
unsloth-zoo 2026.4.6 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.5.4 which is incompatible.
5.5.0


In [ ]:
import numpy as np
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def evaluate_model(dataset, model, tokenizer, n=50):
    preds = []
    refs = []

    samples = dataset["test"].select(range(min(n, len(dataset["test"]))))

    for sample in samples:
        # ===== Prompt =====
        prompt = sample["text"].split("<|start_header_id|>assistant<|end_header_id|>")[0]
        prompt += "<|start_header_id|>assistant<|end_header_id|>\n"

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        pred = tokenizer.decode(output[0], skip_special_tokens=True)
        pred = pred.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()

        ref = sample["text"].split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()

        preds.append(pred)
        refs.append(ref)

    return preds, refs

In [ ]:
pip uninstall -y transformers unsloth unsloth-zoo accelerate

Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0
Found existing installation: unsloth 2026.4.4
Uninstalling unsloth-2026.4.4:
  Successfully uninstalled unsloth-2026.4.4
Found existing installation: unsloth_zoo 2026.4.6
Uninstalling unsloth_zoo-2026.4.6:
  Successfully uninstalled unsloth_zoo-2026.4.6
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0


Replace `your_library_name` with the actual name of the library you want to install. For example, to install the `numpy` library, you would use `!pip install numpy`.

In [ ]:
import torch, gc, os

gc.collect()
torch.cuda.empty_cache()

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
pip uninstall -y transformers unsloth unsloth-zoo accelerate trl

Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0
Found existing installation: unsloth 2026.4.4
Uninstalling unsloth-2026.4.4:
  Successfully uninstalled unsloth-2026.4.4
Found existing installation: unsloth_zoo 2026.4.6
Uninstalling unsloth_zoo-2026.4.6:
  Successfully uninstalled unsloth_zoo-2026.4.6
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: trl 0.24.0
Uninstalling trl-0.24.0:
  Successfully uninstalled trl-0.24.0


In [ ]:
!pip install -q unsloth
!pip install -q unsloth-zoo
!pip install -q accelerate datasets evaluate trl bitsandbytes